# EEG-Flow Dataset Size Report

Reports sizes for:
- Raw data: full inventory by file type, all vs included participants
- `derivatives_v2` — canonical version for submission
- `derivatives` — v1, archived reference only

Last edit: 19.10.2026 @anguyen

In [1]:
from datetime import datetime
from pathlib import Path
import sys

def find_repo_root(marker='core') -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / marker).is_dir():
            return candidate
    raise RuntimeError(f"Could not locate repo root (no '{marker}/' found above {Path.cwd()})")

sys.path.insert(0, str(find_repo_root() / 'core'))
from participants import get_participants

if sys.platform == 'win32':
    ACTIVE_ROOT  = Path('L:/EEG_Flow_data/Study_EEG_Flow_Active')
    PASSIVE_ROOT = Path('L:/EEG_Flow_data/Study_EEG_Flow_Passive')
else:
    ACTIVE_ROOT  = Path('/Volumes/faculty/EEG_Flow_data/Study_EEG_Flow_Active')
    PASSIVE_ROOT = Path('/Volumes/faculty/EEG_Flow_data/Study_EEG_Flow_Passive')

# included participant IDs (use load as reference — largest included set)
ACTIVE_IDS  = list(get_participants('load', 'active').keys())
PASSIVE_IDS = list(get_participants('load', 'passive').keys())

print(f'Active included:  {len(ACTIVE_IDS)}')
print(f'Passive included: {len(PASSIVE_IDS)}')


# ==============================================================================
# Helper functions
# ==============================================================================

def folder_size_gb(path: Path) -> float:
    if not path.exists():
        return 0.0
    return sum(f.stat().st_size for f in path.rglob('*') if f.is_file()) / 1e9


def size_by_ext(folders: list[Path], exts: list[str]) -> float:
    return sum(
        f.stat().st_size for sub in folders
        for f in sub.rglob('*') if f.suffix in exts
    ) / 1e9


def game_videos_gb(folders: list[Path]) -> float:
    total = 0.0
    for sub in folders:
        beh = sub / 'beh'
        if beh.exists():
            total += sum(f.stat().st_size for f in beh.rglob('*.mkv')
                         if 'game' in f.name.lower()) / 1e9
    return total


def face_videos_gb(folders: list[Path]) -> float:
    total = 0.0
    for sub in folders:
        beh = sub / 'beh'
        if beh.exists():
            total += sum(f.stat().st_size for f in beh.rglob('*.mkv')
                         if 'face' in f.name.lower()) / 1e9
    return total


def get_sub_folders(root: Path, included_ids: list[str]):
    all_subs  = [p for p in (root / 'raw').iterdir()
                  if p.is_dir() and p.name.startswith('sub-')]
    incl_subs = [p for p in all_subs
                  if any(uid in p.name for uid in included_ids)]
    excl_subs = [p for p in all_subs if p not in incl_subs]
    return all_subs, incl_subs, excl_subs


print('Functions ready.')

Active included:  48
Passive included: 47
Functions ready.


## Raw data inventory

In [2]:
def raw_inventory(label: str, root: Path, included_ids: list[str]) -> dict:
    all_subs, incl_subs, excl_subs = get_sub_folders(root, included_ids)

    col = {'all': all_subs, 'incl': incl_subs, 'excl': excl_subs}
    n   = {k: len(v) for k, v in col.items()}

    def row(label, fn, *args):
        vals = {k: fn(v, *args) for k, v in col.items()}
        print(f"  {label:<35} {vals['all']:>7.1f}GB  {vals['incl']:>10.1f}GB  {vals['excl']:>10.1f}GB")
        return vals

    print(f"\n{'='*70}")
    print(f"  {label}")
    print(f"{'='*70}")
    print(f"  {'':35} {'All':>7}    {'Included':>10}    {'Excluded':>10}")
    print(f"  {'':35} {'N='+str(n['all']):>7}    {'N='+str(n['incl']):>10}    {'N='+str(n['excl']):>10}")
    print(f"  {'-'*68}")

    xdf  = row('EEG recordings (.xdf)',       size_by_ext, ['.xdf'])
    log  = row('Gamelogs (.log)',              size_by_ext, ['.log'])
    gvid = {k: game_videos_gb(v) for k, v in col.items()}
    fvid = {k: face_videos_gb(v) for k, v in col.items()}
    print(f"  {'Game videos (.mkv)':<35} {gvid['all']:>7.1f}GB  {gvid['incl']:>10.1f}GB  {gvid['excl']:>10.1f}GB")
    print(f"  {'Face videos (.mkv) [identifiable]':<35} {fvid['all']:>7.1f}GB  {fvid['incl']:>10.1f}GB  {fvid['excl']:>10.1f}GB")

    print(f"  {'-'*68}")
    for k in ['all', 'incl', 'excl']:
        no_face = xdf[k] + log[k] + gvid[k]
        full    = no_face + fvid[k]
        tag = {'all': 'All', 'incl': 'Included', 'excl': 'Excluded'}[k]
        print(f"  {tag} — EEG only:                      {xdf[k]:>7.1f}GB")
        print(f"  {tag} — EEG + game videos + logs:      {no_face:>7.1f}GB")
        print(f"  {tag} — Full (incl. face video):        {full:>7.1f}GB")

    return {
        'n_all': n['all'], 'n_incl': n['incl'], 'n_excl': n['excl'],
        'xdf': xdf, 'log': log, 'gvid': gvid, 'fvid': fvid
    }


active_raw  = raw_inventory('ACTIVE — Raw data inventory', ACTIVE_ROOT,  ACTIVE_IDS)
passive_raw = raw_inventory('PASSIVE — Raw data inventory', PASSIVE_ROOT, PASSIVE_IDS)


  ACTIVE — Raw data inventory
                                          All      Included      Excluded
                                         N=76          N=48          N=28
  --------------------------------------------------------------------
  EEG recordings (.xdf)                 131.8GB        84.6GB        47.2GB
  Gamelogs (.log)                         0.0GB         0.0GB         0.0GB
  Game videos (.mkv)                    507.3GB       328.6GB       178.7GB
  Face videos (.mkv) [identifiable]     186.9GB       120.8GB        66.0GB
  --------------------------------------------------------------------
  All — EEG only:                        131.8GB
  All — EEG + game videos + logs:        639.1GB
  All — Full (incl. face video):          826.0GB
  Included — EEG only:                         84.6GB
  Included — EEG + game videos + logs:        413.2GB
  Included — Full (incl. face video):          534.0GB
  Excluded — EEG only:                         47.2GB
  Excluded

In [3]:
# Combined raw totals
print(f"\n{'='*70}")
print(f"  BOTH STUDIES — Raw data totals")
print(f"{'='*70}")
for k, tag in [('all', 'All'), ('incl', 'Included'), ('excl', 'Excluded')]:
    xdf  = active_raw['xdf'][k]  + passive_raw['xdf'][k]
    log  = active_raw['log'][k]  + passive_raw['log'][k]
    gvid = active_raw['gvid'][k] + passive_raw['gvid'][k]
    fvid = active_raw['fvid'][k] + passive_raw['fvid'][k]
    n    = active_raw[f'n_{k}']  + passive_raw[f'n_{k}']
    print(f"\n  {tag} (N={n}):")
    print(f"    EEG only:                  {xdf:6.1f} GB")
    print(f"    EEG + game videos + logs:  {xdf+log+gvid:6.1f} GB")
    print(f"    Full (incl. face video):   {xdf+log+gvid+fvid:6.1f} GB")


  BOTH STUDIES — Raw data totals

  All (N=136):
    EEG only:                   235.7 GB
    EEG + game videos + logs:  1153.4 GB
    Full (incl. face video):   1545.5 GB

  Included (N=95):
    EEG only:                   168.1 GB
    EEG + game videos + logs:   824.3 GB
    Full (incl. face video):   1108.3 GB

  Excluded (N=41):
    EEG only:                    67.6 GB
    EEG + game videos + logs:   329.1 GB
    Full (incl. face video):    437.2 GB


## derivatives_v2 breakdown

In [4]:
def report_derivatives(label: str, root: Path) -> dict:
    v2_root  = root / 'derivatives_v2'
    v2_total = folder_size_gb(v2_root)

    pp_size = sum(
        folder_size_gb(p) for p in v2_root.iterdir()
        if p.is_dir() and p.name.startswith('sub-')
    ) if v2_root.exists() else 0.0

    ga   = folder_size_gb(v2_root / 'outputs' / 'grand_averages')
    meas = folder_size_gb(v2_root / 'outputs' / 'measures')
    stat = folder_size_gb(v2_root / 'outputs' / 'stats')
    img  = folder_size_gb(v2_root / 'outputs' / 'img')
    v1   = folder_size_gb(root / 'derivatives')

    print(f"\n{'='*70}")
    print(f"  {label}")
    print(f"{'='*70}")
    print(f"  derivatives_v2 (total):        {v2_total:6.1f} GB  [CANONICAL]")
    print(f"    ├── per-participant (steps):  {pp_size:6.1f} GB")
    print(f"    ├── grand_averages:           {ga:6.1f} GB")
    print(f"    ├── measures:                 {meas:6.1f} GB")
    print(f"    ├── stats:                    {stat:6.1f} GB")
    print(f"    └── img:                      {img:6.1f} GB")
    print(f"  derivatives v1 (total):        {v1:6.1f} GB  [archived]")
    return {'v2': v2_total, 'pp': pp_size, 'ga': ga,
            'meas': meas, 'stat': stat, 'img': img, 'v1': v1}


active_deriv  = report_derivatives('ACTIVE — Derivatives', ACTIVE_ROOT)
passive_deriv = report_derivatives('PASSIVE — Derivatives', PASSIVE_ROOT)


  ACTIVE — Derivatives
  derivatives_v2 (total):         253.1 GB  [CANONICAL]
    ├── per-participant (steps):   252.1 GB
    ├── grand_averages:              0.8 GB
    ├── measures:                    0.1 GB
    ├── stats:                       0.0 GB
    └── img:                         0.0 GB
  derivatives v1 (total):        1465.8 GB  [archived]

  PASSIVE — Derivatives
  derivatives_v2 (total):         238.7 GB  [CANONICAL]
    ├── per-participant (steps):   238.0 GB
    ├── grand_averages:              0.6 GB
    ├── measures:                    0.0 GB
    ├── stats:                       0.0 GB
    └── img:                         0.0 GB
  derivatives v1 (total):         640.8 GB  [archived]


## Grand totals

In [5]:
total_raw_all  = (folder_size_gb(ACTIVE_ROOT / 'raw') +
                  folder_size_gb(PASSIVE_ROOT / 'raw'))
total_v2       = active_deriv['v2']  + passive_deriv['v2']
total_v1       = active_deriv['v1']  + passive_deriv['v1']

print(f"{'='*70}")
print(f"  GRAND TOTALS (both studies)")
print(f"{'='*70}")
print(f"  Raw (all participants):    {total_raw_all:6.1f} GB")
print(f"  derivatives_v2:            {total_v2:6.1f} GB")
print(f"  derivatives v1:            {total_v1:6.1f} GB")
print(f"  Grand total:               {total_raw_all + total_v2 + total_v1:6.1f} GB")

  GRAND TOTALS (both studies)
  Raw (all participants):    1902.9 GB
  derivatives_v2:             491.8 GB
  derivatives v1:            2106.6 GB
  Grand total:               4501.3 GB


## Save report to file

In [6]:
timestamp = datetime.now().strftime('%Y-%m-%d %H:%M')
date_str  = datetime.now().strftime('%Y%m%d')

lines = [
    f'EEG-Flow Dataset Size Report',
    f'Generated: {timestamp}',
    '',
    f'ACTIVE  — N included: {active_raw["n_incl"]} / {active_raw["n_all"]}',
    f'PASSIVE — N included: {passive_raw["n_incl"]} / {passive_raw["n_all"]}',
    '',
    '=== RAW DATA ===',
    f'Active  raw (all):      {folder_size_gb(ACTIVE_ROOT / "raw"):6.1f} GB',
    f'Passive raw (all):      {folder_size_gb(PASSIVE_ROOT / "raw"):6.1f} GB',
    f'Both    raw (all):      {total_raw_all:6.1f} GB',
    '',
    '=== derivatives_v2 [CANONICAL] ===',
    f'Active:                 {active_deriv["v2"]:6.1f} GB',
    f'Passive:                {passive_deriv["v2"]:6.1f} GB',
    f'Both:                   {total_v2:6.1f} GB',
    '',
    '=== derivatives v1 [archived] ===',
    f'Active:                 {active_deriv["v1"]:6.1f} GB',
    f'Passive:                {passive_deriv["v1"]:6.1f} GB',
    f'Both:                   {total_v1:6.1f} GB',
    '',
    '=== GRAND TOTAL ===',
    f'All data:               {total_raw_all + total_v2 + total_v1:6.1f} GB',
]

report = '\n'.join(lines)

for root in [ACTIVE_ROOT, PASSIVE_ROOT]:
    out_path = root / 'derivatives_v2' / 'outputs' / f'data_size_report_{date_str}.txt'
    out_path.write_text(report, encoding='utf-8')
    print(f'Saved -> {out_path}')

Saved -> L:\EEG_Flow_data\Study_EEG_Flow_Active\derivatives_v2\outputs\data_size_report_20261019.txt
Saved -> L:\EEG_Flow_data\Study_EEG_Flow_Passive\derivatives_v2\outputs\data_size_report_20261019.txt
